# Week 4: Transfer Learning, BERT (Homework)

## Question Search Engine

Embeddings are a good source of information for solving various tasks. For example, we can classify texts or find similar documents using their representations. We already know about word2vec, GloVe and fasttext, but they don't use context information from given text (only from contexts of source data).

For today we will use full power of context-aware embeddings to find text duplicates!

__Warning:__ this task assumes you have seen `seminar.ipynb`!

## Rules

This notebook is checked automatically by **ThetaGrader (TG)**.

- Keep every `### Task N` heading intact -- TG uses it as a marker to locate tasks. Do not remove, rename, renumber, or duplicate these headings.
- Write your solution in the cell(s) reserved for it, directly below the corresponding task statement. Do not move a solution to a different place in the notebook or merge several tasks' solutions together.
- If a task asks for a written conclusion, report, or answer to a theoretical question, put it in a markdown cell directly under that task, clearly separated from code.
- Before submitting your final solution, make sure the `Status` field in the Bot-check cell below is set to `final`.

## AI usage disclosure

If you used AI assistants (e.g. ChatGPT, Claude, GitHub Copilot) while working on this homework, briefly describe here which tool(s) you used, for which parts of the assignment, and how.

Leave this cell as "no AI assistance used" if you did not use any.

I used Claude in chat regime. For task 2, I asked it about Hugging Face libraries interface.

In [ ]:
%pip install --upgrade transformers datasets accelerate deepspeed
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
import datasets

### Data Preparation

In [2]:
qqp = datasets.load_dataset("SetFit/qqp")
print("\n")
print("Sample[0]:", qqp["train"][0])
print("Sample[3]:", qqp["train"][3])

README.md:   0%|          | 0.00/313 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train.jsonl: reconstructing file:   0%|          |  0.00B / 70.8MB            

train.jsonl: downloading bytes:           |  0.00B            

validation.jsonl:   0%|          | 0.00/7.83M [00:00<?, ?B/s]

...

test.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/363846 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/40430 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/390965 [00:00<?, ? examples/s]



Sample[0]: {'text1': 'How is the life of a math student? Could you describe your own experiences?', 'text2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0, 'label_text': 'not duplicate'}
Sample[3]: {'text1': 'What can one do after MBBS?', 'text2': 'What do i do after my MBBS ?', 'label': 1, 'idx': 3, 'label_text': 'duplicate'}


In [ ]:
model_name = "gchhablani/bert-base-cased-finetuned-qqp"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/320 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  433MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  433MB            

model.safetensors: downloading bytes:           |  0.00B            

In [4]:
MAX_LENGTH = 128

def preprocess_function(examples):
    result = tokenizer(
        examples["text1"],
        examples["text2"],
        padding="max_length",
        max_length=MAX_LENGTH,
        truncation=True,
    )

    result["label"] = examples["label"]

    return result

In [5]:
qqp_preprocessed = qqp.map(preprocess_function, batched=True)

Map:   0%|          | 0/363846 [00:00<?, ? examples/s]

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

Map:   0%|          | 0/390965 [00:00<?, ? examples/s]

In [6]:
print(repr(qqp_preprocessed["train"][0]["input_ids"])[:100], "...")

[101, 1731, 1110, 1103, 1297, 1104, 170, 12523, 2377, 136, 7426, 1128, 5594, 1240, 1319, 5758, 136,  ...


### Evaluation (1 point)

We randomly chose a model trained on QQP - but is it any good?

One way to measure this is with validation accuracy - which is what you will implement next.

Here's the interface to help you do that:

In [7]:
val_set = qqp_preprocessed["validation"]
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=1, shuffle=False, collate_fn=transformers.default_data_collator
)

In [8]:
for batch in val_loader:
    break  # here be your training code
print("Sample batch:", batch)

with torch.no_grad():
    predicted = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        token_type_ids=batch["token_type_ids"],
    )

print("\nPrediction (probs):", torch.softmax(predicted.logits, dim=1).data.numpy())

Sample batch: {'labels': tensor([0]), 'idx': tensor([0]), 'input_ids': tensor([[  101,  2009,  1132,  2170,   118,  4038,  1177,  2712,   136,   102,
          2009,  1132,  1117, 10224,  4724,  1177,  2712,   136,   102,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,   

### Task 1 (1 point):

- Measure the validation accuracy of your model. Doing so naively may take several hours. Please make sure you use the following optimizations:
  - Run the model on GPU with no_grad
  - Using batch size larger than 1
  - Use optimize data loader with num_workers > 1
  - (Optional) Use [mixed precision](https://pytorch.org/docs/stable/notes/amp_examples.html)

In [13]:
# <A whole lot of YOUR CODE HERE>
from tqdm import tqdm

def validate_model(model, val_set):
    val_loader = torch.utils.data.DataLoader(
        val_set, batch_size=64, shuffle=False, collate_fn=transformers.default_data_collator, num_workers=2
    )

    model.half().cuda().eval()
    
    total_right = 0
    with torch.no_grad():
        for batch in tqdm(val_loader):
            batch = {k: v.to("cuda") for k, v in batch.items()}
            predicted = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                token_type_ids=batch["token_type_ids"],
            )

            predictions = torch.argmax(predicted.logits, dim=1)
            total_right += (predictions == batch["labels"]).sum().item()
    return total_right / len(val_set)

accuracy = validate_model(model, qqp_preprocessed["validation"])

100%|██████████| 632/632 [00:55<00:00, 11.41it/s]


In [14]:
assert 0.9 < accuracy < 0.91

### Training (4 points)

For this task, you have two options:

__Option A:__ fine-tune your own model. You are free to choose any model __except for the original BERT.__ We recommend [DeBERTa-v3](https://huggingface.co/microsoft/deberta-v3-base). Better yet, choose the best model based on public benchmarks (e.g. [GLUE](https://gluebenchmark.com/)).

You can write the training code manually or use transformers.Trainer (see [this example](https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification)). Please make sure that your model's accuracy is at least __comparable__ with the above example for BERT.


__Option B:__ compare at least 3 pre-finetuned models (in addition to the above BERT model). For each model, report (1) its accuracy, (2) its speed, measured in samples per second in your hardware setup and (3) its size in megabytes. Please take care to compare models in equal setting, e.g. same CPU / GPU. Compile your results into a table and write a short (~half-page on top of a table) report, summarizing your findings.

### Task 2 (4 points):
- Choose Option A or Option B (only one will be graded)
- Follow all the instructions and restrictions

In [ ]:
%pip install evaluate
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments, Trainer
import evaluate
import numpy as np


model_name = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
deberta = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2, dtype=torch.float32)

MAX_LENGTH = 256
def tokenize(examples):
    result = tokenizer(
        examples["text1"],
        examples["text2"],
        max_length=MAX_LENGTH,
        truncation=True,
    )
    result["label"] = examples["label"]
    return result

qqp_tokenized = qqp.map(tokenize, batched=True).remove_columns(["text1", "text2"])
train_set = qqp_tokenized["train"]
val_set = qqp_tokenized["validation"]

def compute_metrics(eval_pred):
    accuracy_metric = evaluate.load("accuracy")
    f1_metric = evaluate.load("f1")
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"],
        "f1": f1_metric.compute(predictions=predictions, references=labels)["f1"],
    }

collator = DataCollatorWithPadding(tokenizer=tokenizer)

args = TrainingArguments(
    output_dir="debertav3-qqp",
    num_train_epochs=2,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    fp16=True,
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    learning_rate=5e-5,
    warmup_steps=0.1,
    logging_first_step=True,
    logging_strategy="steps",
    logging_steps=1000,
    eval_on_start=True,
    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="epoch",  
    save_total_limit=2,
    hub_strategy="every_save"
)

trainer = Trainer(
    model=deberta,
    args=args,
    data_collator=collator,
    train_dataset=train_set,
    eval_dataset=val_set,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

# trainer.train()

# from huggingface_hub import login
# login()
# trainer.push_to_hub()

MAX_LENGTH = 256
model_name = "ma-tt-a/debertav3-qqp"
tokenizer = AutoTokenizer.from_pretrained(model_name)
deberta = AutoModelForSequenceClassification.from_pretrained(model_name)

def preprocess_function(examples):
    result = tokenizer(
        examples["text1"],
        examples["text2"],
        padding="max_length",
        max_length=MAX_LENGTH,
        truncation=True,
    )
    result["label"] = examples["label"]
    return result

qqp_preprocessed = qqp.map(preprocess_function, batched=True)
val_set = qqp_preprocessed["validation"]
accuracy = validate_model(deberta, val_set)
print("Deberta validation accuracy:", accuracy)

config.json:   0%|          | 0.00/941 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/538 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/8.34M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  738MB            

model.safetensors: downloading bytes:           |  0.00B            

...

Map:   0%|          | 0/363846 [00:00<?, ? examples/s]

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

Map:   0%|          | 0/390965 [00:00<?, ? examples/s]

100%|██████████| 632/632 [04:18<00:00,  2.45it/s]

Deberta validation accuracy: 0.9217660153351471


We see that our fine-tuned model has achieved better performance on the validation set than the model from Task 1.

### Finding Duplicates (1 point)

Finally, it is time to use your model to find duplicate questions.
Please implement a function that takes a question and finds top-5 potential duplicates in the training set. For now, it is fine if your function is slow, as long as it yields correct results.

Showcase how your function works with at least 5 examples.

### Task 3 (1 point):
- Implement function for finding duplicates
- Test it on several examples (at least 5)
- Check suggested duplicates and make a conclusion about model correctness

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np


def preprocess_for_inference(examples):
    return {
        "text": examples["text1"] + examples["text2"]
    }


def collate_inference(query, batch):
    texts = [sample["text"] for sample in batch]
    return tokenizer(
        [query["text"]] * len(texts),
        texts,
        max_length=MAX_LENGTH,
        truncation=True,
        padding=True,
        return_tensors="pt",
        )

def find_scores(model, query, batch_size=256):
    collate_fn = lambda batch: collate_inference(query, batch)

    loader = DataLoader(
        qqp_for_inference,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn,
    )

    scores = []
    model.half().cuda().eval()
    with torch.no_grad():
        for batch in tqdm(loader):
            batch = {k: v.to("cuda") for k, v in batch.items()}
            logits = model(**batch).logits
            scores += logits.float().cpu().numpy()[:, 1].tolist()

    return np.array(scores)


def find_topk(model, queries, k=5):
    topk_results = []
    for query in queries:
        scores = find_scores(model, query)
        topk_indices = np.argsort(scores)[-k:][::-1]
        topk_results.append(qqp_for_inference[topk_indices])
    for query, results in zip(queries, topk_results):
        print(f"Query: {query['text']}")
        for i, result in enumerate(results["text"]):
            print(f"Top {i+1}: {result}")
        print()
        
    return topk_results

qqp_for_inference = qqp['train'].map(preprocess_for_inference, batched=True, remove_columns=qqp['train'].column_names)

queries = [
    {"text": "How to study quantum mechanics effectively?"},
    {"text": "What is it like to be a mathematics student?"},
    {"text": "What is your favorite Shakespeare play?"},
    {"text": "Do you read Illiad and Odyssey?"},
    {"text": "What is the best way to learn linear algebra?"}
]

MAX_LENGTH = 256
model_name = "ma-tt-a/debertav3-qqp"
tokenizer = AutoTokenizer.from_pretrained(model_name)
deberta = AutoModelForSequenceClassification.from_pretrained(model_name)
model = torch.nn.DataParallel(deberta) 
scores = find_topk(model, queries, k=3)

Map:   0%|          | 0/363846 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/941 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/538 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/8.34M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  738MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

100%|██████████| 2843/2843 [11:23<00:00,  4.16it/s]


Query: How to study quantum mechanics effectively?
Top 1: How do I study quantum mechanics well?
Top 2: I want to study Quantum Physics, where do I start?
Top 3: How can I study more effectively in mbbs?

Query: What is it like to be a mathematics student?
Top 1: How is the life of a math student? Could you describe your own experiences?
Top 2: How is the mytacs experience?
Top 3: What is height og SSC CGL preparation what is height of SSC CGL preparation?

Query: What is your favorite Shakespeare play?
Top 1: Which were the greatest Shakespeare plays?
Top 2: Are Chinese characters inefficient? Quora Top Stories from Your Feed Your Quora Digest Are Chinese characters inefficient
Top 3: Why do Police is not accepting bribe online? Why they want bribe in cash? Our PM want us to be Cashless! Then why Police want bribe in cash?

Query: Do you read Illiad and Odyssey?
Top 1: What is height og SSC CGL preparation what is height of SSC CGL preparation?
Top 2: Are Chinese characters inefficien

Based on the 5 examples above and top-3 duplicate candidates for each, we can observe that the model performs pretty well. For the first three requests the model has found almost identical sentences. For the last query it has found sentence with similar structure. It has failed on the 4th query, but it can be explained by that there is not similar questions in train set at all.

### Bonus: Finding Duplicates Faster (0.5 point)

Try to find a way to run the function faster than just passing over all questions in a loop. For isntance, you can form a short-list of potential candidates using a cheaper method, and then run your tranformer on that short list. If you opted for this solution, please keep both the original implementation and the optimized one - and explain briefly what is the difference there.

### Task 4 (Bonus, 0.5 point):
- Speed up your implementation from "Finding Duplicates" part
- Capture both old and new implementation work time
- Describe your approach

In [ ]:
<A whole lot of YOUR CODE HERE>

### Bonus: Finding Duplicates in Old-Fashioned way (1.5 points)

In this bonus task you are supposed to use pretrained embeddings (word2vec, GloVe or fasttext) for solving the duplicates problem.

### Task 5 (Bonus, 1.5 points):
- Solve Finding Duplicates problem using mentioned embeddings
- Compare old-fashioned solution to previous ones (quality, speed, etc.)
- Make a small report (up to 5 steps, results and conclusions) on work done in this part

In [ ]:
<A whole lot of YOUR CODE HERE>